In [ ]:
pip install requests bs4 transformers gradio


In [ ]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline
import gradio as gr

# Initialize the summarization pipeline
summarizer = pipeline("summarization")

def summarize_text(text):
    try:
        # Summarize the text
        summary = summarizer(text, max_length=150, min_length=40, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        return f"Error in summarization: {e}"

# Topics to choose from
topics = ["India", "World", "Business", "Tech", "Cricket", "Sports", "Entertainment", "Auto", "TV"]

def fetch_and_summarize_news(selected_topic):
    url = f"https://timesofindia.indiatimes.com/topic/{selected_topic}"
    response = requests.get(url)
    results = []

    if response.status_code == 200:
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        # Find elements with class 'uwU81' (Limit to first 5 articles)
        elements = soup.find_all(class_='uwU81', limit=5)

        scraped_data = []
        for element in elements:
            text = element.get_text(strip=True)
            link = element.find('a')['href'] if element.find('a') else None
            if link:
                scraped_data.append({'text': text, 'link': link})

        for item in scraped_data:
            link_response = requests.get(item['link'])
            if link_response.status_code == 200:
                link_soup = BeautifulSoup(link_response.content, 'html.parser')
                article_text = link_soup.get_text(strip=True)
                short_text = article_text[:2000]
                summary = summarize_text(short_text)
                summary = summary.split('|')[0]

                # Format the result as a mini "card" without the title
                result_str = f"""<div class="news-card">
                    <p><strong>Link:</strong> <a href="{item['link']}" target="_blank" class="news-link">Read Full Article</a></p>
                    <p class="news-summary">{summary}</p>
                </div>"""
                results.append(result_str)
            else:
                results.append(f"Failed to scrape link: {item['link']} with status code {link_response.status_code}")
    else:
        results.append(f"Failed to retrieve the webpage. Status code: {response.status_code}")

    # Join all results and wrap in a container
    return "<div class='results-container'>" + "".join(results) + "</div>"

# Custom CSS for styling
custom_css = """
body {
    margin:0;
    padding:0;
}

.gradio-container {
    background: linear-gradient(135deg, #f0f9ff 0%, #e0f7fa 100%);
    font-family: 'Helvetica', 'Arial', sans-serif;
    color: #333;
}

#header-title {
    font-size: 2.5em;
    font-weight: bold;
    color: #333;
    text-align: center;
    margin-top: 40px;
    margin-bottom: 10px;
}

#header-subtitle {
    font-size: 1.2em;
    color: #555;
    text-align: center;
    margin-bottom: 30px;
    line-height: 1.5;
}

#description {
    text-align: center;
    font-size: 1.1em;
    max-width: 600px;
    margin: 0 auto 30px auto;
    color: #444;
}

.label {
    color: #4CAF50 !important;
    font-weight: bold !important;
    font-size: 1.1em !important;
    margin-bottom: 5px !important;
}

button {
    background-color: #4CAF50 !important;
    color: white !important;
    border-radius: 4px !important;
    border: none !important;
    padding: 12px 24px !important;
    font-size: 16px !important;
    cursor: pointer !important;
    margin-top: 10px !important;
    margin-bottom: 20px !important;
    transition: background-color 0.3s ease !important;
}

button:hover {
    background-color: #45a049 !important;
}

#output_box {
    background-color: #fff;
    color: #333;
    border: none;
    border-radius: 8px;
    padding: 20px;
    font-family: sans-serif;
    line-height: 1.5;
    box-shadow: 0 2px 8px rgba(0,0,0,0.1);
    min-height: 300px;
    overflow: auto;
}

.results-container {
    display: flex;
    flex-direction: column;
    gap: 20px;
}

.news-card {
    background: #ffffff;
    border-radius: 8px;
    padding: 20px;
    border: 1px solid #ddd;
    box-shadow: 0 1px 4px rgba(0,0,0,0.1);
}

.news-link {
    color: #4CAF50;
    text-decoration: none;
    font-weight: bold;
}
.news-link:hover {
    text-decoration: underline;
}

.news-summary {
    margin-top: 15px;
    font-size: 1em;
    line-height: 1.4;
    color: #444;
    white-space: pre-wrap;
    word-wrap: break-word;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    # Header section
    gr.HTML("<h1 id='header-title'>📰 Times of India Summarizer</h1>")
    gr.HTML("<p id='header-subtitle'>Quickly catch up on the latest news headlines in just a click!</p>")

    # Description / Instruction
    gr.HTML("<p id='description'>✨ Ready for a quick news fix? ✨<br>Select a topic below, then hit 'Summarize' to instantly fetch & condense the hottest headlines! 📰🔥<br><strong>It's that easy!</strong></p>")

    with gr.Row():
        topic_input = gr.Radio(choices=topics, label="Select a Topic:")
    summarize_button = gr.Button("Summarize")

    # We will render HTML output directly to leverage styling
    output_box = gr.HTML(elem_id="output_box")

    summarize_button.click(fn=fetch_and_summarize_news, inputs=topic_input, outputs=output_box)

if __name__ == "__main__":
    demo.launch()


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a6c05a07440f47298.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
